# Installations

In [15]:
!pip install -q langchain
!pip install -q langchain-community
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q transformers
!pip install -q accelerate
!pip install -q pymupdf
!pip install -q pypdf
!pip install -q google-genai

In [6]:
from google.colab import files
uploaded = files.upload()

Saving Document Question Answering System (RAG)..pdf to Document Question Answering System (RAG)..pdf
Saving image net.pdf to image net (1).pdf


uploaded two files ImageNet Classification with Deep Convolutional
Neural Networks, PDF Retrieval Augmented Question Answering documentations

# Read the PDFs and Extract the text

In [7]:
import fitz
documents = []
for pdf_file in uploaded.keys():
    print(f"Reading: {pdf_file}")

    doc = fitz.open(pdf_file)
    text = ""

    for page in doc:
        text += page.get_text()

    documents.append(text)
print("\nTotal Documents:", len(documents))
print("\nPreview:\n")
print(documents[0][:500])

Reading: Document Question Answering System (RAG)..pdf
Reading: image net (1).pdf

Total Documents: 2

Preview:

PDF Retrieval Augmented Question Answering
Thi Thu Uyen Hoang
Meenakshi Rajendran
Kun Zhang
Yuhan Wu
Viet Anh Nguyen
Saarland University
{thho00003, mera00002, kuzh00001, yuwu00001, ving00001}@stud.uni-saarland.de
Abstract
This paper presents an advancement in Question-Answering (QA) systems using
a Retrieval Augmented Generation (RAG) framework to enhance information
extraction from PDF files. Recognizing the richness and diversity of data within
PDFs—including text, images, vector diagrams, gr


# Text to Chunks

In [10]:
!pip install -q -U langchain-text-splitters

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = []

for doc in documents:
    chunks.extend(splitter.split_text(doc))

print("Total Chunks:", len(chunks))
print("\nFirst Chunk:\n")
print(chunks[0])

Total Chunks: 199

First Chunk:

PDF Retrieval Augmented Question Answering
Thi Thu Uyen Hoang
Meenakshi Rajendran
Kun Zhang
Yuhan Wu
Viet Anh Nguyen
Saarland University
{thho00003, mera00002, kuzh00001, yuwu00001, ving00001}@stud.uni-saarland.de
Abstract
This paper presents an advancement in Question-Answering (QA) systems using
a Retrieval Augmented Generation (RAG) framework to enhance information
extraction from PDF files. Recognizing the richness and diversity of data within


# Embeddings

In [12]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)
embeddings = embedding_model.encode(
    chunks,
    show_progress_bar=True,
    convert_to_numpy=True
)
print("Embedding Shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Embedding Shape: (199, 384)


# Creating Vector Database

In [13]:
import faiss
import numpy as np
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype("float32"))
print("FAISS index created successfully!")
print("Number of vectors stored:", index.ntotal)

FAISS index created successfully!
Number of vectors stored: 199


# Loading API Key

API Key is mentioned in secrets.

In [16]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
if GEMINI_API_KEY:
    print("API Key Loaded Successfully")
else:
    print("API Key Not Found")

API Key Loaded Successfully


In [17]:
from google import genai
client = genai.Client(api_key=GEMINI_API_KEY)
print("Gemini Client Initialized")

Gemini Client Initialized


# Retrieval Function

In [18]:
import numpy as np

def retrieve(query, top_k=3):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")
    distances, indices = index.search(query_embedding, top_k)

    retrieved_chunks = []

    for idx in indices[0]:
        retrieved_chunks.append(chunks[idx])

    return retrieved_chunks

In [24]:
def answer_question(question):

    retrieved_docs = retrieve(question)

    context = "\n\n".join(retrieved_docs)

    prompt = f"""
You are a helpful AI assistant.

Answer the user's question ONLY using the provided context.

If the answer is not available in the context, reply:

"I couldn't find that information in the uploaded documents."

Context:
{context}

Question:
{question}

Answer:
"""

    response = client.models.generate_content(
        model="gemini-flash-latest",
        contents=prompt
    )

    return response.text

In [22]:
from google import genai

client = genai.Client(api_key=GEMINI_API_KEY)

for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-omni-flash-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
mod

# Questioning

In [25]:
while True:

    question = input("\nAsk a Question (type 'exit' to stop): ")

    if question.lower() == "exit":
        print("Session Ended.")
        break

    answer = answer_question(question)

    print("\n" + "="*60)
    print("Question:")
    print(question)

    print("\nAnswer:")
    print(answer)

    print("="*60)


Ask a Question (type 'exit' to stop): what is the dataset in Imagenet document

Question:
what is the dataset in Imagenet document

Answer:
Based on the provided context, ImageNet is a dataset of over 15 million labeled high-resolution images belonging to roughly 22,000 categories (also described as over 22,000 categories). The images were collected from the web and labeled by human labelers. 

The context also mentions a Fall 2009 version of ImageNet, which consists of 10,184 categories and 8.9 million images.

Ask a Question (type 'exit' to stop): what are two documents about

Question:
what are two documents about

Answer:
I couldn't find that information in the uploaded documents.

Ask a Question (type 'exit' to stop): who are the authors of PDF Retrieval Augmented Question Answering

Question:
who are the authors of PDF Retrieval Augmented Question Answering

Answer:
The authors of "PDF Retrieval Augmented Question Answering" are:

* Thi Thu Uyen Hoang
* Meenakshi Rajendran
* Kun

### Conclusion

The RAG system successfully answers questions when the required information is explicitly available in the uploaded PDF documents. It retrieves relevant content and generates accurate, context-based responses. However, it does not perform well for broad questions such as summarizing the entire PDF, as it focuses only on the most relevant retrieved sections.
